# Lesson 02 - Explorando o Microsoft Agent Framework

O **Microsoft Agent Framework (MAF)** é um framework unificado para construir agentes de IA. Fornece uma arquitetura limpa e componível com quatro blocos principais:

- **Client** – conecta-se a um endpoint de modelo de IA e gere a comunicação
- **Agent** – envolve um cliente com instruções e definições de ferramentas
- **Tools** – estendem as capacidades do agente com funções personalizadas que o modelo pode chamar
- **Session** – mantém o histórico da conversa para interações de múltiplas etapas

Nesta lição, vamos construir um **agente de reserva de viagens** que verifica a disponibilidade do destino utilizando estes conceitos.


## Configuração


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Compreender a Arquitetura do Agent Framework

O Microsoft Agent Framework segue uma arquitetura em camadas:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Cliente** – Um `AzureAIProjectAgentProvider` liga-se a uma implementação Azure OpenAI. Trata da autenticação, formatação dos pedidos e análise das respostas.
2. **Agente** – Criado a partir do cliente através de `provider.create_agent()`, o agente combina o acesso ao modelo com instruções (prompt do sistema) e ferramentas.
3. **Ferramentas** – Funções Python decoradas com `@tool` que o agente pode invocar para realizar ações ou obter dados.
4. **Sessão** – Um objeto `AgentSession` (criado através de `agent.create_session()`) que armazena o historial da conversa, permitindo diálogo em múltiplas interações onde o agente lembra o contexto anterior.

Vamos construir cada camada passo a passo.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Adicionar Ferramentas com o Decorador @tool

As ferramentas permitem aos agentes realizar ações para além de gerar texto. O decorador `@tool` converte uma função Python comum numa coisa que o agente pode chamar.

Pontos chave:
- Use `Annotated[type, "description"]` para que o modelo compreenda cada parâmetro.
- A docstring torna-se na descrição da ferramenta que o modelo vê.
- `approval_mode="never_require"` significa que a ferramenta é executada automaticamente sem confirmação do utilizador.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Criar um Agente com Ferramentas

Agora combinamos o cliente, as instruções e as ferramentas num agente. As `instructions` funcionam como o prompt do sistema — definem a persona e o comportamento do agente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversas Multi-Turnos com Sessões

Uma `AgentSession` (criada via `agent.create_session()`) mantém o registo de todas as mensagens numa conversa. Ao passar a mesma sessão para cada chamada `agent.run()`, o agente tem acesso a todo o histórico da conversa e pode referir-se a mensagens anteriores.

Passamos `tools=[check_destination_availability]` para que o agente possa chamar o nosso verificador de disponibilidade durante cada turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Sumário

Nesta aula explorou os quatro pilares do Microsoft Agent Framework:

| Conceito | O Que Aprendeu |
|---------|------------------|
| **Cliente** | `AzureAIProjectAgentProvider` liga-se ao Azure OpenAI com autenticação baseada em credenciais |
| **Agente** | `provider.create_agent()` agrupa uma ligação a um modelo com instruções e um nome |
| **Ferramentas** | O decorador `@tool` expõe funções Python para o agente invocar |
| **Sessão** | `agent.create_session()` mantém o histórico da conversa ao longo de múltiplos turnos |

Estes blocos construtivos combinam-se para criar agentes que podem manter conversas naturais, invocar funções externas e manter o contexto — a base para padrões agentes mais avançados em aulas posteriores.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos para garantir a precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deverá ser considerado a fonte oficial. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações erradas decorrentes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
